# Notebook 12: BERT multilingüe sin palabras de veredicto

Comprobación de la fuga de etiqueta en el modelo principal. En el corpus ecuatoriano la etiqueta se asignó buscando palabras de veredicto ("falso", "engañoso", "cierto", "califica...") en el título y en los primeros 1,000 caracteres del contenido, y el modelo se entrena con ese mismo contenido.

Aquí se repite **exactamente** la validación cruzada del notebook 10 (mismo modelo, mismos hiperparámetros, mismas 5 particiones con semilla 42), con **un solo cambio**: se quitan esas palabras del texto antes de tokenizar.

**Cómo usarlo en Colab:** activar la GPU T4 (Entorno de ejecución > Cambiar tipo de entorno de ejecución), subir `corpus_ecuador_final.csv` al panel de archivos y ejecutar la celda. Tarda unos 20 minutos. No exporta ningún modelo y no modifica `bert_ecuador`.

In [2]:
import numpy as np, pandas as pd, torch, time
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score, recall_score
from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup

MODEL_NAME = "bert-base-multilingual-cased"
MAX_LENGTH, BATCH_SIZE, EPOCHS, LR, SEED = 256, 16, 4, 2e-5, 42
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

df = pd.read_csv("corpus_ecuador_final.csv")        # el mismo archivo del notebook 08
import re
# ÚNICO CAMBIO respecto al notebook 10: se quitan del texto las palabras de veredicto
# con las que el scraper asignó la etiqueta (falso, engañoso, cierto, califica, etc.).
VEREDICTO = r"\b(?:falso|falsa|enga[ñn]oso|enga[ñn]osa|cierto|cierta|verdadero|verdadera|impreciso|imprecisa|califica|calific[oó])\b"
originales = df["contenido"].fillna("").tolist()
X = [re.sub(VEREDICTO, " ", t, flags=re.I) for t in originales]
cambiados = sum(a != b for a, b in zip(originales, X))
print(f"Textos modificados: {cambiados} de {len(X)}")
y = df["label_binario"].values
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class DS(Dataset):
    def __init__(self, textos, etiquetas):
        self.enc = tokenizer(textos, max_length=MAX_LENGTH, padding="max_length",
                             truncation=True, return_tensors="pt")
        self.y = torch.tensor(etiquetas, dtype=torch.long)
    def __len__(self):
        return len(self.y)
    def __getitem__(self, i):
        item = {k: v[i] for k, v in self.enc.items()}
        item["labels"] = self.y[i]
        return item

def predecir(model, loader):
    model.eval(); preds = []
    with torch.no_grad():
        for b in loader:
            out = model(input_ids=b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE))
            preds.extend(out.logits.argmax(dim=1).cpu().numpy())
    return np.array(preds)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
resultados = []
for fold, (tr, te) in enumerate(cv.split(X, y), start=1):
    t0 = time.time()
    torch.manual_seed(SEED); np.random.seed(SEED)
    dl_tr = DataLoader(DS([X[i] for i in tr], y[tr]), batch_size=BATCH_SIZE, shuffle=True)
    dl_te = DataLoader(DS([X[i] for i in te], y[te]), batch_size=BATCH_SIZE)

    model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2).to(DEVICE)
    conteo = np.bincount(y[tr], minlength=2)
    pesos = torch.tensor(len(tr) / (2 * conteo), dtype=torch.float).to(DEVICE)   # pesos por clase
    loss_fn = torch.nn.CrossEntropyLoss(weight=pesos)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
    total = len(dl_tr) * EPOCHS
    sch = get_linear_schedule_with_warmup(opt, int(0.1 * total), total)

    for ep in range(EPOCHS):
        model.train()
        for b in dl_tr:
            opt.zero_grad()
            out = model(input_ids=b["input_ids"].to(DEVICE), attention_mask=b["attention_mask"].to(DEVICE))
            loss_fn(out.logits, b["labels"].to(DEVICE)).backward()
            opt.step(); sch.step()

    pred = predecir(model, dl_te)
    r = dict(fold=fold, acc=accuracy_score(y[te], pred),
             f1_macro=f1_score(y[te], pred, average="macro"),
             rec_ver=recall_score(y[te], pred, pos_label=1))
    resultados.append(r)
    print(f"Fold {fold}: acc={r['acc']:.3f}  F1 macro={r['f1_macro']:.3f}  "
          f"recall Verificado={r['rec_ver']:.3f}  ({(time.time() - t0) / 60:.1f} min)", flush=True)
    del model, opt; torch.cuda.empty_cache()

res = pd.DataFrame(resultados)
f = lambda c: f"{res[c].mean():.3f}±{res[c].std(ddof=0):.3f}"
print("\nBERT multilingüe + pesos por clase, 5 particiones:")
print(f"acc={f('acc')}  F1 macro={f('f1_macro')}  recall Verificado={f('rec_ver')}")

Textos modificados: 1504 de 1543


config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  714MB            

... (12 outputs left out)

Fold 4: acc=0.851  F1 macro=0.711  recall Verificado=0.545  (3.8 min)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 5: acc=0.828  F1 macro=0.713  recall Verificado=0.682  (3.9 min)

BERT multilingüe + pesos por clase, 5 particiones:
acc=0.832±0.024  F1 macro=0.715±0.023  recall Verificado=0.659±0.058


## Comparación con el notebook 10

In [3]:
# Cifras del notebook 10 (BERT con texto completo, mismas particiones)
referencia = {"acc": 0.843, "f1_macro": 0.735, "rec_ver": 0.695}
print(f"{'Métrica':<22}{'Texto completo (nb 10)':>24}{'Sin veredicto (este)':>24}{'Diferencia':>14}")
for clave, nombre in (("acc", "Accuracy"), ("f1_macro", "F1 macro"), ("rec_ver", "Recall Verificado")):
    ahora = res[clave].mean()
    print(f"{nombre:<22}{referencia[clave]:>24.3f}{ahora:>24.3f}{ahora - referencia[clave]:>+14.3f}")
print("\nClasificador trivial: F1 macro 0.461, recall de Verificado 0.000")
print("Regresión logística (chequeo previo): F1 macro 0.718 -> 0.651, recall 0.480 -> 0.390")

Métrica                 Texto completo (nb 10)    Sin veredicto (este)    Diferencia
Accuracy                                 0.843                   0.832        -0.011
F1 macro                                 0.735                   0.715        -0.020
Recall Verificado                        0.695                   0.659        -0.036

Clasificador trivial: F1 macro 0.461, recall de Verificado 0.000
Regresión logística (chequeo previo): F1 macro 0.718 -> 0.651, recall 0.480 -> 0.390


## Cómo leer el resultado

- Si el F1 macro y el recall casi no cambian respecto al notebook 10, el BERT no depende de esas palabras.
- Si bajan mucho, parte de su desempeño viene de que el texto contiene el veredicto.
- En cualquier caso, la caída se compara con la del clasificador trivial (F1 macro 0.461) y con la de la regresión logística (de 0.718 a 0.651). Quitar solo esas palabras no elimina otras pistas del estilo de las verificaciones, así que el resultado es un piso de la fuga, no un techo.